In [3]:
import ast
import pandas as pd

# Path ke file dataset
train_path = "../../data/raw/semeval2021/data/tsd_train.csv"
trial_path = "../../data/raw/semeval2021/data/tsd_trial.csv"
test_path  = "../../data/raw/semeval2021/data/tsd_test.csv"

# Baca CSV dengan converter ast.literal_eval
df_train = pd.read_csv(train_path, converters={"spans": ast.literal_eval})
df_trial = pd.read_csv(trial_path, converters={"spans": ast.literal_eval})
df_test  = pd.read_csv(test_path, converters={"spans": ast.literal_eval})

print(f"Data Train Counts : {len(df_train)}")
print(f"Data Trial Counts : {len(df_trial)}")
print(f"Data Test Counts  : {len(df_test)}")
# Cek tipe data kolom spans
print("Tipe data spans:", type(df_train["spans"].iloc[0])) # Output: <class 'list'>
print(">>> Dataframe train:")
df_train.head()

Data Train Counts : 7939
Data Trial Counts : 690
Data Test Counts  : 2000
Tipe data spans: <class 'list'>
>>> Dataframe train:


,spans,text
0,"[8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,...",Another violent and aggressive immigrant killi...
1,"[33, 34, 35, 36, 37, 38, 39]","I am 56 years old, I am not your fucking junio..."
2,"[0, 1, 2, 3]","Damn, a whole family. Sad indeed."
3,"[7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]",What a knucklehead. How can anyone not know th...
4,"[32, 33, 34, 35, 36, 37, 38]","""who do you think should do the killing?""\n\nA..."


In [3]:
from itertools import groupby
from operator import itemgetter

def extract_toxic_words(text, spans):
    """
    Mengubah list indeks karakter [15, 16, 17, 18, 19]
    menjadi teks potongan aslinya.
    """
    if not spans:
        return []

    # Kelompokkan indeks yang berurutan
    words = []
    for _, g in groupby(enumerate(sorted(spans)), lambda ix: ix[0] - ix[1]):
        group = list(map(itemgetter(1), g))
        start, end = group[0], group[-1] + 1
        words.append(text[start:end])
    return words

# Contoh melihat 5 sampel pertama beserta kata toksiknya
for i in range(5):
    row = df_train.iloc[i]
    toxic_words = extract_toxic_words(row["text"], row["spans"])

    print(f"--- Data ke-{i+1} ---")
    print(f"Teks Asli   : {row['text']}")
    print(f"Spans Indeks: {row['spans'][:10]}... (Total {len(row['spans'])} karakter)" if len(row['spans']) > 10 else f"Spans: {row['spans']}")
    print(f"Kata Toksik : {toxic_words}")
    print()

--- Data ke-1 ---
Teks Asli   : Another violent and aggressive immigrant killing a innocent and intelligent US Citizen.... Sarcasm
Spans Indeks: [8, 9, 10, 11, 12, 13, 14, 15, 16, 17]... (Total 32 karakter)
Kata Toksik : ['violent and aggressive immigrant']

--- Data ke-2 ---
Teks Asli   : I am 56 years old, I am not your fucking junior pal. What you are saying makes no sense. I don't know what you are basing this on. The cheap black market crap is still coming up from Mexican nationals, and the good stuff from BC and Humbolt.  The I5 corridor is still just as busy, and they are still making a ton of money. Nothing has really changed except that the legal market has made it easy for anyone of legal age to buy it.  And the legal market is controlled by corporate growers with million dollar facilities, or a co-op of smaller growers. The Federal government "war on drugs" really has no impact on the legal market in Oregon. I don't see any glutton of weed, whatever that means.
Spans: [33, 3